In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'tensorflow', 'keras', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'keras': 'keras==3.14.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "061a82116be00c585cab41d5b2a790e2a4f96d44"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/061a82116be00c585cab41d5b2a790e2a4f96d44/d2l"
for name in ('__init__.py', 'tensorflow.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Stochastic Differential Equations

The two previous sections integrated deterministic vector fields. From a given
start the trajectory is fixed, and a flow never forgets where it came from:
two different starting points always reach two different endpoints. A
continuous normalizing flow can still carry a data distribution to a
Gaussian, but only after it has been trained on that particular data. The
diffusion models of the main text work the other way round. Their forward
process adds fresh noise at every step until nothing of the starting point
remains, and this is what makes generation possible, because the reverse
process then has a known place to begin. What equation does such a process
solve, and how do we calculate with it when the noise is new at every instant?

A small example shows what is involved. Take one number, $x_0 = 2$. At each
step keep $99\%$ of it and add an independent Gaussian draw with standard
deviation $\sqrt{0.02} \approx 0.14$:

$$
x_{k+1} = 0.99\, x_k + \sqrt{0.02}\; \xi_k, \qquad \xi_k \sim \mathcal{N}(0, 1).
$$

After $400$ steps the mean is $2 \cdot 0.99^{400} \approx 0.036$ and the
variance is close to $1$.
Any other start ends in the same place, as a draw from a distribution very
close to $\mathcal{N}(0, 1)$. One feature of the recurrence looks odd. If it
describes a process in continuous time with time step $h = 0.01$, then the
decay $0.99 = 1 - h$ changes $x$ by an amount of order $h$, but the noise has
size $\sqrt{h}$, not $h$. This choice is forced. Noise of size $h$ would
vanish as $h \to 0$, while noise of size $\sqrt{h}$ survives, and its square is of
order $h$, the same order as the drift. Ordinary calculus discards squares of
small changes, so this one fact requires a new calculus.

The continuous-time limit of such noise is Brownian motion, named after the
botanist Robert Brown, who watched pollen grains jitter in water in 1827.
Norbert Wiener constructed it as a mathematical object in 1923, and Kiyoshi
Itô built the calculus for equations driven by it in the 1940s
[@Ito.1944]. The central
idea of Itô's calculus is that along a Brownian path the square of a short
increment equals the elapsed time on average, so the chain rule gains a
second-derivative term.

The forward process of a diffusion model is such an equation, and
that section is where the book uses it: the
variance-preserving path there is the process behind our recurrence. Its law
at each time, given the start, is Gaussian, which gives denoising score
matching its regression targets in closed form. We take the limit of random
walks, build the calculus, solve the process in closed form, and then
simulate general equations and measure the error.

Three results carry the section, and the rest is preparation or consequence:
the quadratic variation $(dW)^2 = dt$, Itô's lemma, and the Gaussian kernel
of the Ornstein--Uhlenbeck process. Only the first is hard to believe, and
its proof takes six lines. A reader in a hurry can take the two properties of
the Itô integral (that section) on trust and leave the
orders of convergence (that section) for a second
reading.
Throughout, $f$ is the drift, $g$ the noise amplitude, $W$ Brownian motion,
$\theta$ the decay rate of the running example, and $\alpha_t$, $\sigma_t$ the
mean factor and the standard deviation of its Gaussian marginals.

In [ ]:
import math
import numpy as np
import tensorflow as tf
from d2l import tensorflow as d2l

## Brownian Motion

### Noise as a Forward Process

Before building any machinery we check what a forward process needs, and why
a deterministic flow cannot provide it. Starting from data
distributed as $p_0 = p_{\textrm{data}}$, the process must end at one known
distribution, whatever the data looked like,

$$
p_0 = p_{\textrm{data}} \;\longrightarrow\; p_T \approx \mathcal{N}(\mathbf{0}, \mathbf{I}),
$$

so that at generation time we always know where to start. A deterministic flow
cannot do this for every dataset at once. The flows of
that section are invertible maps $\Phi$, and an invertible
map sends distinct laws to distinct laws: if $\Phi$ carried $p$ and $q$ to the
same distribution, applying $\Phi^{-1}$ would show $p = q$. So no single fixed
flow carries two different datasets to the same endpoint.

Let us try three processes on two "datasets" that are as simple as datasets
can be: point masses at $x_0 = 2$ and at $x_0 = -2$.

- **Drift alone**, $\dot{x} = -x$. The solution is $x(t) = x_0 e^{-t}$, so at
  $t = 3$ the two starts sit at $\pm 2e^{-3} = \pm 0.0996$. Both datasets
  shrink toward the point mass at $0$, which is not a Gaussian, and at every
  finite time they are still two distinct points.
- **Noise alone**, $dX = \sqrt{2}\, dW$. Here $W$ is a noise whose variance
  grows linearly in time, constructed in the next subsection, and the equation
  means $X_t = x_0 + \sqrt{2}\, W_t$. The law at time $t$ is
  $\mathcal{N}(x_0, 2t)$: each cloud keeps its center at $\pm 2$ while its
  standard deviation $\sqrt{2t}$ grows without bound ($2.45$ at $t = 3$). The
  start is forgotten only relative to this growing width, and there is no
  limiting law.
- **Both together**, $dX = -X\,dt + \sqrt{2}\, dW$. From either start the law
  at $t = 3$ is $\mathcal{N}(\pm 0.0996,\, 0.9975)$, a formula proved later in
  this section. The two laws are nearly indistinguishable from each other and
  from $\mathcal{N}(0, 1)$.

![Two failures and the remedy, from the two starts $x_0=\pm2$. Drift alone ($\dot x=-x$) sends every start to the point $0$. Noise alone ($X_t=x_0+\sqrt2\,W_t$) keeps each start as the center of a cloud whose two-standard-deviation envelope $x_0\pm2\sqrt{2t}$ grows without bound. Both together ($dX=-X\,dt+\sqrt2\,dW$) carry every start into the same stationary law $\mathcal N(0,1)$, drawn sideways at the right. Green lines are the means; dashed lines are the analytic two-standard-deviation envelopes.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-dyn-sde-paths.svg)

the figure draws the three processes. Only the first
fails outright. The second is usable, and is in use, if it runs until its
variance $2t$ is far larger than that of the data; this family is called
*variance-exploding*, and its endpoint variance must be made huge because it
has no stationary law. The third is the running example of this section, and
the recurrence of the opening simulates it. Its drift pulls each start toward
zero while the noise refills the variance that the drift removes, and the two
balance at variance $1$. that section proves that
this happens from every start.

In general, a forward process of this kind is a deterministic vector field
plus a noise term. For data in $\mathbb{R}^d$ we write

$$
d\mathbf{X} \;=\; \underbrace{\mathbf{f}(\mathbf{X}, t)\,dt}_{\textrm{drift}}
\;+\; \underbrace{g(t)\,d\mathbf{W}}_{\textrm{diffusion}},
$$

which reads: over a short time $dt$, the state moves by the velocity
$\mathbf{f}$ times $dt$ and receives a random kick whose size is set by the
noise amplitude $g$. The drift steers the mean, and the diffusion injects
variance. Such a process can also be run backward in distribution, which is
how a diffusion model generates; the next section shows how.

### From Random Walks to the Wiener Process

The noise term $d\mathbf{W}$ needs a precise object behind it. We build it as
the limit of the simplest random process, a coin-flip walk, and the limit
explains the square root in our recurrence.

Fix a step duration $\Delta t$ and let a walker take independent steps of size
$\pm\sqrt{\Delta t}$, each sign chosen by a fair coin. After $k$ steps, at time
$t = k\,\Delta t$, the position is a sum of $k$ independent mean-zero steps of
variance $\Delta t$ apiece. Variances of independent terms add
(that section), so

$$
\mathbb{E}\!\left[W^{(\Delta t)}_t\right] = 0,
\qquad
\textrm{Var}\!\left(W^{(\Delta t)}_t\right) = k\,\Delta t = t,
$$

whatever the step size. With $\Delta t = 0.01$, for instance, $100$ steps of
$\pm 0.1$ give variance $100 \cdot 0.01 = 1$ at time $1$. Steps of $\pm 0.01$
would give variance $0.01$, and steps of $\pm 1$ would give $100$.

More generally, steps of size $c\,\Delta t^{a}$ give variance
$(t/\Delta t)\, c^2 \Delta t^{2a} = c^2\, t\, \Delta t^{2a - 1}$ at time $t$,
which tends to $0$ for $a > \tfrac12$ and to $\infty$ for $a < \tfrac12$.
Among such step sizes only $a = \tfrac12$ gives a finite nonzero variance in
the limit, and this square root returns in the noise term of every numerical
scheme below. As $\Delta t \to 0$ the number of steps before time $t$ grows
without bound, and the central limit theorem (a sum of many independent
mean-zero terms with total variance $t$ is approximately $\mathcal{N}(0, t)$)
makes the limiting position Gaussian.

The limit process is called **Brownian motion** or the **Wiener process**
[@Wiener.1923]. It is defined by four properties:

1. $W_0 = 0$;
2. **independent increments**: for $s < t$, the increment $W_t - W_s$ is
   independent of the entire path up to time $s$;
3. **Gaussian increments**: $W_t - W_s \sim \mathcal{N}(0,\, t - s)$;
4. continuous sample paths.

That a process with these properties exists is Wiener's theorem, and that the
rescaled walks converge to it is Donsker's theorem; we take both as given. In
particular $\mathbb{E}[W_t] = 0$ and $\textrm{Var}(W_t) = t$: the process
spreads with standard deviation $\sqrt{t}$ forever
(the figure). Simulation uses the increment form

$$
\Delta W \;=\; W_{t + \Delta t} - W_t \;=\; \sqrt{\Delta t}\;\xi,
\qquad \xi \sim \mathcal{N}(0, 1),
$$

which says that a Brownian increment over a step of length $\Delta t$ is a
standard normal draw scaled by $\sqrt{\Delta t}$.

![One coin-flip walk of $1000$ steps $\pm\sqrt{0.001}$, viewed on meshes $\Delta t=0.1$, $0.01$ and $0.001$ (the same randomness; faint gray: the full path). Each view is itself a random walk with step variance $\Delta t$; refining the mesh fills in detail without changing the large-scale path. At each fixed time the spread follows the $\pm\sqrt t$ and $\pm2\sqrt t$ envelopes.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-dyn-brownian-paths.svg)

The multivariate Brownian motion $\mathbf{W}_t$ runs one independent scalar
Brownian motion per coordinate. For the diagonal, state-independent noise
$g(t)\,d\mathbf{W}$ of the diffusion-model SDEs in this chapter, every
calculation below therefore applies coordinate by coordinate.

Independent increments determine how the values of $W$ at two times are
related. For $s \le t$ split $W_t = W_s + (W_t - W_s)$; the increment is
independent of $W_s$ and both have mean zero, so

$$
\textrm{Cov}(W_s, W_t)
= \mathbb{E}\!\left[W_s^2\right] + \mathbb{E}[W_s]\,\mathbb{E}[W_t - W_s]
= s = \min(s, t).
$$

The walk carries its position at time $s$ forward unchanged and adds fresh
noise that is uncorrelated with it. The formula is worth more than its
one-line proof suggests: a Gaussian process is determined by its mean and
covariance, so mean zero, covariance $\min(s, t)$ and continuous paths
characterize Brownian motion.

The same increments make the path rough. Brownian paths are continuous but
nowhere differentiable, a theorem of Paley, Wiener, and Zygmund
[@Paley.Wiener.Zygmund.1933]. The increment scaling
the equation gives the heuristic reason:

$$
\left|\frac{\Delta W}{\Delta t}\right|
= \frac{|\xi|}{\sqrt{\Delta t}}
\;\longrightarrow\; \infty
\quad \textrm{in probability as } \Delta t \to 0,
$$

so a difference quotient grows without bound as the step shrinks. This is not
a proof (it rules out a derivative at each fixed time, while "nowhere"
concerns all times at once), but it shows the obstruction.

Rescaling time by a factor $c$ and space by $\sqrt{c}$ returns a Brownian
motion (problem 12), so zooming
in on a path shows the same roughness at every scale, where a differentiable
curve would flatten into its tangent. Hence $dW/dt$ does not exist, and $dW$
has to be given a meaning inside integrals. The Itô calculus supplies that
meaning.

### Simulation of Brownian Paths

The increment form the equation is also a simulation recipe:
take cumulative sums of $\sqrt{\Delta t}\,\xi$ steps. We generate $20{,}000$
paths on $[0, 1]$ this way, and with them the random generator `rng` that
every later cell draws from.

In [ ]:
rng = np.random.default_rng()
T_bm, n_bm, n_paths = 1.0, 500, 20000
dt_bm = T_bm / n_bm
dW_bm = np.sqrt(dt_bm) * rng.standard_normal((n_paths, n_bm))
W_bm = np.concatenate([np.zeros((n_paths, 1)), np.cumsum(dW_bm, axis=1)],
                      axis=1)
for t in [0.25, 0.5, 1.0]:
    k = round(t / dt_bm)
    print(f'Var(W_t) at t={t:.2f}: {W_bm[:, k].var():.4f}   (theory: {t:.2f})')
inside = np.abs(W_bm[:, -1]) < 2 * np.sqrt(T_bm)
print(f'fraction inside +-2 sqrt(t) at t=1: {inside.mean():.3f}'
      '   (Gaussian: 0.954)')

The variances equal $t$ up to the Monte Carlo standard error
$t\sqrt{2/20000}$, or $1\%$ of $t$, as they must for sums of such steps: the
printout tests the code, not the theory. About $95\%$ of the paths lie inside
$\pm 2\sqrt{t}$ at $t = 1$. This is a statement about one fixed time. Far
fewer paths, about half on this grid, stay inside the envelope over the whole
interval. The same ensemble gives the covariance, as the average of
$W_s W_t$ over a grid of time pairs.

In [ ]:
ts = np.array([0.1, 0.25, 0.5, 0.75, 1.0])
idx = np.round(ts / dt_bm).astype(int)
emp = W_bm[:, idx].T @ W_bm[:, idx] / n_paths   # E[W_s W_t]; means are zero
print('empirical E[W_s W_t]:')
print(emp.round(3))
print('max |empirical - min(s,t)| = '
      f'{np.abs(emp - np.minimum.outer(ts, ts)).max():.4f}')

Each row is constant to the right of the diagonal, up to a sampling error of
about $0.01$: what happens after time $s$ adds nothing to the covariance with
$W_s$.

## Itô Calculus

### Quadratic Variation

Ordinary calculus stops its Taylor expansions at first order because the
square of a small change is negligible. For Brownian motion this fails, and
the result of this subsection says by how much. It is the one on which the
rest of the section rests.

Take a partition $0 = t_0 < t_1 < \cdots < t_n = t$ with increments
$\Delta t_i = t_{i+1} - t_i$ and mesh $h_{\max} = \max_i \Delta t_i$, and add
up the squared increments of a path along it. For a differentiable path $x$
with $|x'| \le M$, the mean value theorem gives
$|\Delta x_i| \le M\, \Delta t_i$, so

$$
\sum_{i} (\Delta x_i)^2
\;\le\; M^2 \sum_i \Delta t_i^2
\;\le\; M^2\, h_{\max}\, t
\;\longrightarrow\; 0
$$

as the mesh shrinks. This is why $(dx)^2$ is negligible against $dx$ in
ordinary calculus: a smooth increment squared is of order $\Delta t^2$. A
Brownian increment squared is of order $\Delta t$, since
$\mathbb{E}[(\Delta W_i)^2] = \Delta t_i$, and these add up to $t$ no matter how
fine the mesh (the figure). The sum is random, but
its fluctuations vanish in the limit.

**Proposition (quadratic variation of Brownian motion).** *As the mesh
$h_{\max} \to 0$,*

$$
Q_n \;=\; \sum_{i=0}^{n-1} \left(W_{t_{i+1}} - W_{t_i}\right)^2
\;\longrightarrow\; t
\quad \textrm{in mean square:} \quad
\mathbb{E}\!\left[(Q_n - t)^2\right] \to 0.
$$

**Proof.** Each increment is $\mathcal{N}(0, \Delta t_i)$, so
$\mathbb{E}[(\Delta W_i)^2] = \Delta t_i$ and
$\mathbb{E}[Q_n] = \sum_i \Delta t_i = t$ for every partition. For the
spread, a Gaussian has fourth moment $3\,\Delta t_i^2$, so
$\textrm{Var}\!\left((\Delta W_i)^2\right) = 3\Delta t_i^2 - \Delta t_i^2
= 2\Delta t_i^2$; the increments are independent, so their variances add:

$$
\mathbb{E}\!\left[(Q_n - t)^2\right] = \textrm{Var}(Q_n)
= \sum_i 2\,\Delta t_i^2
\;\le\; 2\,h_{\max} \sum_i \Delta t_i = 2\,h_{\max}\, t \;\longrightarrow\; 0,
$$

and mean-square convergence follows. $\blacksquare$

The sum of squared Brownian increments converges to the elapsed time, a
deterministic number, although every increment is random. On a uniform grid
with $n$ intervals of $[0, 1]$ the proof gives the standard deviation
$\sqrt{2/n}$: $0.35$ for $n = 16$ and $0.003$ for $n = 2^{18}$.

![Running sums of squared increments along one Brownian path for meshes of $n = 16$, $256$, and $4096$ intervals (darker shades are finer meshes). The sums approach $y = t$, whereas the corresponding sums for the smooth path $\sin 2\pi t$ (gray) approach zero. The smooth sums shrink like $2\pi^2/n$, which is still $1.2$ at $n=16$, $0.077$ at $256$ and $0.005$ at $4096$.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-dyn-qv-convergence.svg)

The proposition is usually written as the **Itô multiplication table**, the
working rules for products of differentials:

$$
(dW)^2 = dt, \qquad dW\,dt = 0, \qquad (dt)^2 = 0.
$$

The first rule is the equation in shorthand: a squared Brownian
increment contributes at order $dt$. The other two record that
$dW\,dt$, of size $\Delta t^{3/2}$, and $(dt)^2 = \Delta t^2$ vanish faster
than $\Delta t$ and so contribute nothing in the limit.

At fine meshes the fluctuations are so small that a single path shows the
limit. We sample a path on $2^{18}$ intervals of $[0, 1]$, view it on coarser
subgrids, and print its sum of squares next to the predicted standard
deviation, together with the same sum for the smooth path $\sin 2\pi t$ and
its predicted value $2\pi^2/n$.

In [ ]:
n_fine = 2**18
dW_qv = np.sqrt(1.0 / n_fine) * rng.standard_normal(n_fine)
W_qv = np.concatenate([[0.0], np.cumsum(dW_qv)])
print(f'{"n":>7} {"QV of W":>8} {"sqrt(2/n)":>10} {"QV of sin":>10}'
      f' {"2 pi^2/n":>9}')
for k in range(4, 19, 2):
    n = 2**k
    W_n = W_qv[::n_fine // n]                 # the same path on a coarser grid
    s_n = np.sin(2 * np.pi * np.linspace(0, 1, n + 1))
    print(f'{n:7d} {np.sum(np.diff(W_n)**2):8.4f} {np.sqrt(2 / n):10.4f} '
          f'{np.sum(np.diff(s_n)**2):10.6f} {2 * np.pi**2 / n:9.6f}')

The Brownian column scatters around $1$ with the standard deviation
$\sqrt{2/n}$ that the proof predicts. On $16$ intervals it typically lands
between about $0.5$ and $1.6$, and it settles to $t = 1$ within a few
thousandths at $n = 2^{18}$. The smooth column is large on coarse grids,
$1.22$ at $n = 16$, but it shrinks like $2\pi^2/n$ and agrees with that
prediction to three digits from $n = 256$ on.

### The Itô Integral

The rough path forces us to define integrals against $dW$ directly. We start
with the simplest one, $\int_0^t W\,dW$, and find that the usual Riemann-sum
definition does not single out an answer.

For a smooth integrator, Riemann sums with the integrand evaluated at the left
endpoint, the right endpoint, or the midpoint of each interval all converge to
the same limit. For the integrand $H = W$, the right-endpoint sum exceeds the
left-endpoint sum by

$$
\sum_i W_{t_{i+1}}\,\Delta W_i - \sum_i W_{t_i}\,\Delta W_i
= \sum_i \left(W_{t_{i+1}} - W_{t_i}\right)\Delta W_i = \sum_i (\Delta W_i)^2
\;\longrightarrow\; t,
$$

by the proposition just proved. The two discretizations of "the same"
integral disagree by a finite, deterministic amount, here on one path with
$1000$ steps.

In [ ]:
n_lr = 1000
dt_lr = 1.0 / n_lr
t_lr = np.linspace(0, 1, n_lr + 1)
dW_lr = np.sqrt(dt_lr) * rng.standard_normal(n_lr)
W_lr = np.concatenate([[0.0], np.cumsum(dW_lr)])
left = np.sum(W_lr[:-1] * dW_lr)              # integrand at the left endpoint
right = np.sum(W_lr[1:] * dW_lr)              # integrand at the right endpoint
print(f'left sum {left:.4f}   right sum {right:.4f}')
print(f'right - left {right - left:.4f}   sum of (dW)^2 {np.sum(dW_lr**2):.4f}')

The left sum is a random number that changes from run to run. The right sum
exceeds it by exactly the printed sum of squared increments, which usually
lies within about $0.1$ of $1$ (its standard deviation is
$\sqrt{2/1000} \approx 0.045$). We must choose an evaluation point, and the
choice changes the answer.

Itô's choice is the left endpoint, so that the integrand never sees the
coming increment. A fair game explains why. Think of
$\Delta W_i$ as the outcome of the $i$-th fair bet and of $H_i$ as the stake
placed on it. A gambler must choose the stake from what is known at time
$t_i$, before the bet is settled, and the total winnings are
$\sum_i H_i\,\Delta W_i$. Because the increment $\Delta W_i$ is independent of
everything known at $t_i$, including $H_i$, each bet has expected winnings
$\mathbb{E}[H_i\,\Delta W_i] = \mathbb{E}[H_i]\,\mathbb{E}[\Delta W_i] = 0$, so
no betting strategy has positive expected winnings. A stake that could peek at
the outcome, as the right-endpoint sum does, would make money on average.

The definition writes this rule down. An integrand is *predictable* if its
value on each interval is decided before the increment over that interval is
drawn. A **simple predictable process** $H$ is
constant on each interval, $H_s = H_i$ for $s \in (t_i, t_{i+1}]$, where $H_i$
is determined by the path up to time $t_i$ and $\mathbb{E}[H_i^2] < \infty$.
Its **Itô integral** is the sum of the winnings,

$$
\int_0^t H_s \, dW_s
\;=\; \sum_i H_i\left(W_{t_{i+1}}-W_{t_i}\right).
$$

The fair-game argument shows that it has mean zero,
$\mathbb{E}\!\left[\int_0^t H\,dW\right] = 0$. Its variance is computable as
well.

**Proposition (Itô isometry).** *For a simple predictable $H$,*

$$
\mathbb{E}\!\left[\left(\int_0^t H_s\,dW_s\right)^{\!2}\,\right]
\;=\; \int_0^t \mathbb{E}\!\left[H_s^2\right] ds.
$$

**Proof.** Expand the square of the sum in
the equation. The step that needs care is the cross terms.
A cross term with $i < j$ contains the factor $\Delta W_j$, which is
independent of everything known at time $t_j$, including the product
$H_i\,\Delta W_i\,H_j$. So the expectation factorizes:
$\mathbb{E}\!\left[H_i\,\Delta W_i\, H_j\, \Delta W_j\right] =
\mathbb{E}\!\left[H_i\,\Delta W_i\, H_j\right]\mathbb{E}[\Delta W_j] = 0$.
A diagonal term factorizes the same way into
$\mathbb{E}\!\left[H_i^2\right]\mathbb{E}\!\left[(\Delta W_i)^2\right] =
\mathbb{E}\!\left[H_i^2\right]\Delta t_i$. Summing the diagonal gives
$\sum_i \mathbb{E}[H_i^2]\,\Delta t_i = \int_0^t \mathbb{E}[H_s^2]\,ds$.
$\blacksquare$

The isometry is also what carries the definition beyond step processes, to
integrands such as $H = W$ itself. It says that $H \mapsto \int H\,dW$
preserves the norm $\|H\|^2 = \int_0^t \mathbb{E}[H_s^2]\,ds$. So if simple
predictable processes converge to $H$ in this norm, their integrals converge
in mean square, and the limit is taken as the definition of $\int_0^t H\,dW$.
The zero mean and the isometry survive the limit and hold for every
predictable $H$ of finite norm. In particular, the left-endpoint sums of a
continuous integrand of this kind converge to its integral. We do not prove
these statements; see @Oksendal.2003, chapter 3.

The isometry turns the variance of a noise integral into an ordinary integral
of a second moment. For the integral we started from,
$\mathbb{E}[W_s^2] = s$, so

$$
\textrm{Var}\!\left(\int_0^1 W_s\,dW_s\right) = \int_0^1 s\,ds = \tfrac12 .
$$

A sample of $20{,}000$ left-endpoint sums on the same $1000$-step grid shows
the size of the fluctuations.

In [ ]:
dW_iso = np.sqrt(dt_lr) * rng.standard_normal((20000, n_lr))
W_iso = np.cumsum(dW_iso, axis=1) - dW_iso    # W at the left end of each step
I_iso = np.sum(W_iso * dW_iso, axis=1)        # 20,000 draws of int_0^1 W dW
print(f'mean {I_iso.mean():.4f}   (theory 0)')
print(f'variance {I_iso.var():.4f}   (theory 1/2)')

Mean and variance agree with $0$ and $\tfrac12$ within their standard errors,
$0.005$ and $0.013$; the second is larger than for a Gaussian because the
integral has heavy tails. Itô's lemma below gives the same $\tfrac12$ without
sampling.

The other evaluation points define other integrals. The midpoint choice, or
the average of the left and right sums, defines the *Stratonovich integral*
[@Stratonovich.1966]. It obeys the ordinary chain rule but gives up the
zero mean, because its integrand sees half of the coming increment. This book
uses Itô's convention, for two reasons: the zero mean makes expectations
easy, and the simplest simulation step, which evaluates the coefficients at
the start of the step, is an Itô sum
(problem 9).

### Drift, Diffusion, and Solutions

With the integral in hand, the forward-process template
the equation acquires a meaning. A **stochastic differential
equation** is written

$$
dX = f(X, t)\,dt + g(X, t)\,dW,
$$

and the differential form is shorthand for the integral equation

$$
X_t = X_0 + \int_0^t f(X_s, s)\,ds + \int_0^t g(X_s, s)\,dW_s,
$$

an ordinary integral plus an Itô integral. The **drift** $f$ steers the
average motion and the **diffusion** coefficient $g$ sets the size of the
noise. Over a short step, with the coefficients frozen at the start of the
step, $\mathbb{E}[dX] = f\,dt$ and $\textrm{Var}(dX) = g^2\,dt$. So $f$ is the
conditional mean velocity and $g^2$ the rate at which variance is pumped in.
With $g \equiv 0$ the equation is the ODE of that section.

For example, $dX = g\,dW$ with a constant $g$ has no drift integral, and the
Itô integral of a constant is $g\,W_t$, so
$X_t = X_0 + g\,W_t$. This is the noise-only panel of
the figure with $g = \sqrt{2}$.

As for ODEs, Lipschitz coefficients guarantee a solution. Suppose that $f$ and
$g$ are Lipschitz in $x$ and grow at most linearly,
$|f(x, t)| + |g(x, t)| \le C(1 + |x|)$, and that $X_0$ is independent of $W$
with $\mathbb{E}[X_0^2] < \infty$. Then the equation has a unique solution
(@Oksendal.2003, Theorem 5.2.1). The proof is a stochastic version of
the Picard iteration of that section, with the
isometry in place of the integral bound.

We distinguish two kinds of noise. In **additive** noise the amplitude depends
on time only, $g = g(t)$, as in the template the equation; in
**multiplicative** noise it depends on the state, $g = g(X, t)$. The
difference looks cosmetic, but it decides how accurately a simulated path
follows the true one (that section). The
diffusion-model SDEs of this chapter have additive noise, in the two
families of
the figure: noise alone (variance-exploding) and noise
balanced by a restoring drift.

Unlike an ODE solution, a solution of an SDE is not one curve but a
distribution over paths. Running the equation many times from the same start
with fresh noise gives a cloud of paths like those of
the figure, and what a generative model cares about is
the law of the cloud at each time.

### Itô's Lemma

To compute with an SDE we need the chain rule: if $X$ solves an SDE, what
equation does a function $\phi(X_t, t)$ solve? The ordinary chain rule gives
$d(W^2) = 2W\,dW$, an Itô integral with mean zero, but
$\mathbb{E}[W_t^2] = t$. Something is missing.

One step of a walk shows the missing term. Take $\phi(x) = x^2$ and one step
$\pm\sqrt{\Delta t}$ from $x$, each sign with probability $\tfrac12$. The
average of $\phi$ after the step is

$$
\tfrac12\left[(x + \sqrt{\Delta t})^2 + (x - \sqrt{\Delta t})^2\right]
= x^2 + \Delta t
= \phi(x) + \tfrac12\phi''(x)\,\Delta t ,
$$

since $\phi'' = 2$. The rise comes from the square of the step,
$(\Delta W)^2 = \Delta t$, which the ordinary chain rule throws away.

With $x = 0.7$ and $\Delta t = 0.25$ the step lands at $0.2$ or $1.2$, where
$\phi$ is $0.04$ or $1.44$. The average $0.74$ lies above $\phi(0.7) = 0.49$
by $0.25 = \Delta t$ (the figure). A convex function
gains from symmetric noise, and because a squared Brownian increment is of
order $dt$, the gain per unit time, $\tfrac12\phi''g^2$, does not vanish in
the limit.

![The Itô correction as Jensen's gap. One step of $\pm\sqrt{\Delta t}$ from $x$ moves a convex $\phi$ to an average that lies above $\phi(x)$ by $\tfrac12\phi''(x)\,\Delta t$; here $\phi=x^2$, $x=0.7$, $\Delta t=0.25$, and the gap is $0.25$.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-dyn-ito-jensen.svg)

The general formula comes from the same computation. Let $X$ solve
$dX = f\,dt + g\,dW$ and let $\phi(x, t)$ be once continuously differentiable
in time and twice in the state. A small change of $\phi$, expanded to second
order because first order is no longer enough, is

$$
d\phi = \phi_t\,dt + \phi_x\,dX + \tfrac12 \phi_{xx}\,(dX)^2 + \cdots.
$$

Substitute $dX = f\,dt + g\,dW$ and multiply out $(dX)^2$ with the table
the equation:

$$
(dX)^2 = f^2 (dt)^2 + 2 f g\, dt\, dW + g^2 (dW)^2 = g^2\,dt.
$$

The drift and cross terms vanish, and the noise term survives at first
order. Collecting terms gives the chain rule of stochastic calculus. What we
have given is a heuristic derivation, so we state the result as a theorem
without proof.

**Theorem (Itô's lemma).** *Let $X$ solve $dX = f\,dt + g\,dW$ under the
conditions of the previous subsection, and let $\phi \in C^{1,2}$, once
continuously differentiable in time and twice in the state. Then
$Y_t = \phi(X_t, t)$ satisfies*

$$
d\phi \;=\; \left(\phi_t + f\,\phi_x + \tfrac12 g^2\,\phi_{xx}\right) dt
\;+\; g\,\phi_x \, dW.
$$

Relative to the ordinary chain rule, the new term is
$\tfrac12 g^2 \phi_{xx}\,dt$, the **Itô correction**. The proof has to
control the remainder terms of the Taylor expansion in mean square
(@Oksendal.2003, chapter 4).

**Example 1: $W^2$.** Take $X = W$ (so $f = 0$, $g = 1$) and $\phi(x) = x^2$.
The lemma gives $d(W^2) = 2W\,dW + dt$, the $dt$ that the ordinary chain rule
dropped. In integrated form,

$$
W_t^2 = \int_0^t 2 W_s \, dW_s + t,
\qquad\textrm{that is,}\qquad
\int_0^t W_s\,dW_s = \tfrac12\left(W_t^2 - t\right).
$$

The $-t$ restores the zero mean of the Itô integral, since
$\mathbb{E}[W_t^2] = t$. The formula also gives the variance of the integral
a second way: $W_1^2$ is a squared standard normal, with variance $2$, so
$\textrm{Var}\big(\tfrac12(W_1^2 - 1)\big) = \tfrac14 \cdot 2 = \tfrac12$, the
number the isometry gave.

The correction can be watched as it accumulates. On any grid the gap between
$W_t^2$ and the left-endpoint sum for $\int_0^t 2W\,dW$ telescopes,
$W_t^2 - \sum_{t_i < t} 2W_{t_i}\Delta W_i = \sum_{t_i < t} (\Delta W_i)^2$, so it
is the running quadratic variation and should follow the line $t$. We reuse
the path of `#sdes-left-right-sums`.

In [ ]:
ito = np.concatenate([[0.0], np.cumsum(2 * W_lr[:-1] * dW_lr)])  # int 2W dW
gap = W_lr**2 - ito
print(f'final gap W_1^2 - int 2W dW = {gap[-1]:.4f}   (Ito correction: t = 1)')
print(f'max |gap_t - t| along the path = {np.abs(gap - t_lr).max():.4f}')
d2l.plot(t_lr, [W_lr**2, ito, ito + t_lr], 't', 'value',
         legend=['$W_t^2$', 'naive $\\int_0^t 2W\\,dW$',
                 '$\\int_0^t 2W\\,dW + t$'], figsize=(5, 3))

The final gap equals the sum of squared increments printed by the earlier
cell, so it usually lies within about $0.1$ of $1$. Along the whole path the
gap follows $t$ closely, typically within about $0.05$ and rarely by more than
$0.1$: by the proof of the equation its standard deviation at time
$t$ is $\sqrt{2t\,\Delta t}$, which is $0.045$ at $t = 1$ for $1000$ steps and
shrinks as the grid is refined. The naive curve sags below $W_t^2$, and adding
$t$ puts it back on top.

**Example 2: geometric Brownian motion.** The SDE
$dX = \mu X\,dt + \nu X\,dW$ with $X_0 > 0$ models a quantity that grows at
mean rate $\mu$ with noise proportional to its size, such as a stock price
[@Black.Scholes.1973]. Its noise is multiplicative. Apply the lemma to
$\phi(x) = \log x$, with $\phi' = 1/x$, $\phi'' = -1/x^2$, and
$g^2 = \nu^2 x^2$. The logarithm is smooth only for $x > 0$, so assume for
the moment that $X$ stays positive:

$$
d\log X = \left(\mu - \tfrac12\nu^2\right) dt + \nu\, dW,
\qquad\textrm{so}\qquad
X_t = X_0 \exp\!\left(\left(\mu - \tfrac12\nu^2\right) t + \nu W_t\right).
$$

The formula we obtain is positive, and the lemma applied to it as a function
of $W_t$ and $t$ confirms that it solves the equation, which removes the
assumption.
The $-\tfrac12\nu^2$ is the correction with $\phi'' < 0$: a concave function
loses from symmetric noise. Noise therefore lowers the typical growth rate
$\mu - \tfrac12\nu^2$ below the mean growth rate $\mu$, and
problem 4 shows how far the
mean and the median of $X_t$ drift apart. We will measure simulation errors
against this exact solution.

The multivariate form follows the same pattern, and we take it as given
[@Oksendal.2003]. For $d\mathbf{X} = \mathbf{f}\,dt + \mathbf{G}\,d\mathbf{W}$
with $\mathbf{X} \in \mathbb{R}^d$, a matrix $\mathbf{G}$, and multivariate
Brownian motion $\mathbf{W}$, the multiplication table reads
$dW_i\, dW_j = \delta_{ij}\,dt$, and a twice continuously differentiable $\phi$
satisfies

$$
d\phi(\mathbf{X}) \;=\; \nabla\phi \cdot d\mathbf{X}
\;+\; \tfrac12 \sum_{i,j} \left(\mathbf{G} \mathbf{G}^\top\right)_{ij}
\frac{\partial^2\phi}{\partial x_i\, \partial x_j} \; dt .
$$

The correction weights each second derivative by the covariance rate
$\mathbf{G}\mathbf{G}^\top$ of the noise. For the noise
$\mathbf{G} = g(t)\,\mathbf{I}$ of the diffusion-model SDEs it reduces to
$\tfrac12 g(t)^2 \sum_i \partial^2\phi/\partial x_i^2\; dt$, a multiple of the
Laplacian of $\phi$. This term becomes the diffusion term of the Fokker--Planck
equation in the next section.

The lemma needs two continuous derivatives in the state. It does not apply
as stated to a function with a kink, such as $\phi(x) = |x|$.

## The Ornstein--Uhlenbeck Process

The running example now gets its name and its solution. The
Ornstein--Uhlenbeck (OU) process [@Uhlenbeck.Ornstein.1930] is

$$
dX = -\theta X \, dt + g\,dW, \qquad \theta > 0,
$$

the decay ODE $\dot{x} = -\theta x$ of that section
with noise of constant amplitude $g$ added. The drift pulls excursions back
toward zero at rate $\theta$, and the noise keeps producing new ones
(the figure). The recurrence of the opening is
this process with $\theta = 1$ and $g = \sqrt{2}$, advanced on a grid with
step $h = 0.01$ by freezing the drift over each step: one step multiplies by
$1 - \theta h = 0.99$ and adds noise of standard deviation
$g\sqrt{h} = \sqrt{0.02}$.

![Mean reversion. The drift $-\theta x$ (arrows) pulls every path toward $0$ at rate $\theta=1$ while noise of amplitude $g=\sqrt2$ keeps it moving; paths from three starts forget them within a few multiples of $1/\theta$ and then fluctuate inside the stationary band of two standard deviations, $\pm2\sqrt{g^2/2\theta}=\pm2$. Right: the stationary density.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-dyn-ou-mean-reversion.svg)

### The Transition Kernel

The OU process is one of the few SDEs whose law is known in closed form at
every time, and this is why diffusion models are built on it. The law given
the start is Gaussian. Of all the results of this section, this is the one
the rest of the book uses most.

**Proposition (OU transition kernel).** *For the OU process
the equation,*

$$
X_t \mid X_0 = x_0 \;\sim\; \mathcal{N}\!\left(
x_0\, e^{-\theta t},\;
\frac{g^2}{2\theta}\left(1 - e^{-2\theta t}\right)
\right).
$$

**Proof.** We use the integrating factor that solves the decay ODE. Apply
Itô's lemma the equation to $\phi(x, t) = e^{\theta t} x$,
for which $\phi_t = \theta e^{\theta t} x$, $\phi_x = e^{\theta t}$, and
$\phi_{xx} = 0$, so there is no correction:

$$
d\!\left(e^{\theta t} X_t\right)
= \left(\theta e^{\theta t} X_t - \theta e^{\theta t} X_t\right) dt
+ e^{\theta t} g\,dW_t
= g\, e^{\theta t}\,dW_t.
$$

The drift cancels, and integrating from $0$ to $t$ and multiplying by
$e^{-\theta t}$ gives

$$
X_t = X_0\, e^{-\theta t} + g \int_0^t e^{-\theta (t - s)}\, dW_s .
$$

The remaining integral has a deterministic integrand, so it is a limit of
weighted sums of independent Gaussian increments and is Gaussian with mean
zero (we grant that mean-square limits of Gaussians are Gaussian). The Itô
isometry the equation gives its variance:

$$
\textrm{Var}(X_t \mid X_0)
= g^2 \int_0^t e^{-2\theta (t - s)}\, ds
= \frac{g^2}{2\theta}\left(1 - e^{-2\theta t}\right).
\qquad \blacksquare
$$

The solution the equation reads as a fading memory. The
start decays at rate $\theta$, and a noise kick from time $s$ survives with
weight $e^{-\theta(t-s)}$: recent noise counts, old noise is forgotten. In the
recurrence the same thing happens step by step, since a kick from $k$ steps
ago survives with weight $0.99^k \approx e^{-0.01k}$. Every later section,
and that section in the main text, writes the
kernel with two coefficients, the mean factor and the standard deviation,

$$
\alpha_t = e^{-\theta t}, \qquad
\sigma_t^2 = \frac{g^2}{2\theta}\left(1 - e^{-2\theta t}\right),
\qquad\textrm{so that}\qquad
X_t \mid X_0 = x_0 \sim \mathcal{N}\!\left(\alpha_t x_0,\, \sigma_t^2\right).
$$

The data enter only through the mean $\alpha_t x_0$, and the noise only
through the variance $\sigma_t^2$. For the running example ($\theta = 1$,
$g = \sqrt{2}$, $\sigma_t^2 = 1 - e^{-2t}$) started at $x_0 = 2$, the kernel is
$\mathcal{N}(1.213,\, 0.795^2)$ at $t = 0.5$, $\mathcal{N}(0.736,\, 0.930^2)$
at $t = 1$, and $\mathcal{N}(0.037,\, 1.000^2)$ at $t = 4$. Because the kernel is
Gaussian at every time and for every start, everything about the noised
state given its start is known in closed form, including the gradient of its
log-density, $-(x-\alpha_t x_0)/\sigma_t^2$.

The recurrence is only a discretization of the process, so its cloud should
follow the kernel closely but need not follow it exactly. We run it as a
plain loop on $10{,}000$ copies of $x_0 = 2$, save the cloud at five times,
and print its mean and standard deviation next to $2\alpha_t$ and $\sigma_t$,
together with the fraction of the cloud inside $2\alpha_t \pm 2\sigma_t$.

In [ ]:
theta_ou, g_ou, h_ou = 1.0, np.sqrt(2.0), 0.01
X_ou = np.full(10000, 2.0)                    # 10,000 copies of x_0 = 2
snap_steps = {round(t / h_ou): t for t in (0.5, 1.0, 1.5, 2.0, 4.0)}
X_ou_snap = {}
for k in range(1, 401):
    xi = rng.standard_normal(X_ou.size)
    X_ou = (1 - theta_ou * h_ou) * X_ou + g_ou * np.sqrt(h_ou) * xi
    if k in snap_steps:
        X_ou_snap[snap_steps[k]] = X_ou.copy()
print(f'{"t":>4} {"mean":>6} {"2alpha_t":>8} {"std":>6} {"sigma_t":>7}'
      f' {"inside":>7}')
for t, X in X_ou_snap.items():
    alpha_t = np.exp(-theta_ou * t)
    sigma_t = np.sqrt(g_ou**2 / (2 * theta_ou) * (1 - alpha_t**2))
    inside = np.mean(np.abs(X - 2 * alpha_t) < 2 * sigma_t)
    print(f'{t:4.1f} {X.mean():6.3f} {2 * alpha_t:8.3f} {X.std():6.3f} '
          f'{sigma_t:7.3f} {inside:7.3f}')

Means and standard deviations agree with the kernel within two or three
standard errors (about $0.01$ for a mean and $0.007$ for a standard deviation
from $10{,}000$ samples), and about $95\%$ of the cloud lies inside the band
at every time, as for any Gaussian. At this sample size the discretization
cannot be seen. The cloud from the single point $2$ widens quickly and then
stops widening.

### The Stationary Distribution

A forward process must forget its start. Letting $t \to \infty$ in the kernel
the equation shows that the OU process does: the mean
$x_0 e^{-\theta t}$ decays to zero and the variance approaches
$g^2/(2\theta)$, so

$$
X_t \mid X_0 = x_0 \;\longrightarrow\; \mathcal{N}\!\left(0, \, \frac{g^2}{2\theta}\right)
\quad \textrm{in distribution, for every } x_0 .
$$

This is the distribution-independent endpoint that
that section asked for. The limit is also invariant: a
process started in it stays in it. Indeed, if
$X_0 \sim \mathcal{N}(0, g^2/2\theta)$ is independent of $W$, then by
the equation $X_t$ is the sum of $X_0 e^{-\theta t}$ and
an independent Gaussian with mean zero and variance $\sigma_t^2$, hence
Gaussian with mean zero and variance

$$
e^{-2\theta t}\,\frac{g^2}{2\theta}
+ \frac{g^2}{2\theta}\left(1 - e^{-2\theta t}\right)
= \frac{g^2}{2\theta}.
$$

The decay of the start removes as much variance as the noise adds. The
approach to the stationary law happens on the time scale $1/\theta$, the
relaxation time: after a few multiples of it, $e^{-\theta t}$ is negligible.
For the running example $g^2/2\theta = 1$, so the stationary law is
$\mathcal{N}(0, 1)$.

The recurrence has a stationary law of its own, and it is slightly different.
One step maps a variance $v$ to $(1 - \theta h)^2 v + g^2 h$. From $v = 1$ a
step of the running example gives $0.9801 + 0.02 = 1.0001$, slightly more than
$1$, and the fixed point of the map is

$$
v_\infty = \frac{g^2 h}{1 - (1 - \theta h)^2} = \frac{0.02}{0.0199} = 1.005
\qquad (h = 0.01).
$$

The recurrence therefore settles half a percent above the exact variance $1$.
This is the first sign of a discretization error, and
that section shows that it shrinks in proportion to $h$.

The cloud saved at $t = 4$, after four relaxation times, can be compared
with the kernel quantile by quantile.

In [ ]:
X_4 = X_ou_snap[4.0]
z = np.array([-1.96, -0.674, 0.0, 0.674, 1.96])  # N(0, 1) quantiles
q_sim = np.percentile(X_4, [2.5, 25, 50, 75, 97.5])
print('simulated quantiles at t=4:', q_sim.round(2))
print('N(2 e^-4, 1) quantiles:    ', (2 * np.exp(-4) + z).round(2))
a = 1 - theta_ou * h_ou                       # the recurrence's decay factor
print(f'variance {X_4.var():.4f}   (stationary 1, '
      f'recurrence fixed point {g_ou**2 * h_ou / (1 - a**2):.4f})')

The five quantiles agree with those of the Gaussian within a few hundredths,
and the residual mean $2e^{-4} = 0.037$ is still visible in the median. The
half percent, however, is out of reach: the sample variance has a standard
error of $0.014$ and cannot separate $1$ from $1.005$.

### The Variance-Preserving Normalization

The stationary variance $g^2/2\theta$ is a free parameter, and diffusion
models fix it to $1$ by choosing

$$
g^2 = 2\theta
\qquad \Longrightarrow \qquad
X_t \mid X_0 = x_0 \;\longrightarrow\; \mathcal{N}(0, 1).
$$

The running example makes this choice. With it, $\sigma_t^2 = 1 - \alpha_t^2$,
and the coefficient $\bar{\alpha}_t = \alpha_t^2 = e^{-2\theta t}$ of DDPM
[@ho2020denoising] (our $\alpha_t$ is that paper's
$\sqrt{\bar{\alpha}_t}$) lets us write the solution
the equation in distribution as

$$
X_t \;=\; \sqrt{\bar{\alpha}_t}\; X_0 \;+\; \sqrt{1 - \bar{\alpha}_t}\;
\epsilon, \qquad \epsilon \sim \mathcal{N}(0, 1)
\textrm{ independent of } X_0 .
$$

The marginal is a weighted mix of data and fresh noise, with weights whose
squares add to one. Suppose the data have zero mean and unit variance. Taking
variances of the two independent terms gives

$$
\textrm{Var}(X_t)
= \bar{\alpha}_t \textrm{Var}(X_0) + (1 - \bar{\alpha}_t)
= \bar{\alpha}_t + (1 - \bar{\alpha}_t) = 1
\qquad \textrm{for all } t.
$$

This is what **variance-preserving** (VP) means: for unit-variance data the
marginal variance equals $1$ at every time, not only in the limit. The process
moves variance from the data to the noise, a fraction $\bar{\alpha}_t$ still
carried by the data and $1 - \bar{\alpha}_t$ already replaced by noise,
without changing the total. The name describes unit-variance data only. For
data of variance $s^2$ the marginal variance drifts from $s^2$ to $1$
(problem 8), so the scale of
the data matters. Images scaled to $[-1, 1]$, as in
that section, have variance below one, while latent
diffusion models rescale their latents to about unit variance
[@Rombach.Blattmann.Lorenz.ea.2022].

The variance is preserved; the shape of the distribution is not. A
convenient measure is the fourth moment, which is $3$ for a standard
Gaussian. Expanding the equation gives
$\mathbb{E}[X_t^4] = 3 + \bar{\alpha}_t^2\left(\mathbb{E}[X_0^4] - 3\right)$
(same problem). For data that are $\pm1$ with probability $\tfrac12$ each
(the Rademacher distribution), $\mathbb{E}[X_0^4] = 1$, the smallest fourth
moment a unit-variance law can have, and the prediction is
$3 - 2\bar{\alpha}_t^2$: $2.264$ at $t = 0.25$, $2.963$ at $t = 1$, and
$3.000$ at $t = 3$. The cell below runs the recurrence from $50{,}000$ such
points. The recurrence adds a Gaussian $S \sim \mathcal{N}(0, s_n^2)$ to
$a^n X_0$, where $a = 1 - \theta h$ and
$s_n^2 = g^2 h (1 - a^{2n})/(1 - a^2)$, so its own moments are known as well
(columns `rec`).

In [ ]:
X_vp = rng.choice([-1.0, 1.0], size=50000)  # unit variance, far from Gaussian
X_vp_snap = {0: X_vp}
for k in range(1, 301):
    xi = rng.standard_normal(X_vp.size)
    X_vp = a * X_vp + g_ou * np.sqrt(h_ou) * xi
    if k in (25, 100, 300):
        X_vp_snap[k] = X_vp
print(f'{"t":>4} {"Var":>6} {"rec":>6} {"E X^4":>6} {"rec":>6}'
      f' {"3-2abar^2":>9}')
for k, X in X_vp_snap.items():
    s2 = g_ou**2 * h_ou * (1 - a**(2 * k)) / (1 - a**2)  # noise variance so far
    m4_rec = a**(4 * k) + 6 * a**(2 * k) * s2 + 3 * s2**2
    abar = np.exp(-2 * theta_ou * k * h_ou)
    print(f'{k * h_ou:4.2f} {X.var():6.3f} {a**(2 * k) + s2:6.3f} '
          f'{np.mean(X**4):6.3f} {m4_rec:6.3f} {3 - 2 * abar**2:9.3f}')

The variance stays at $1$ within sampling error while the fourth moment
climbs from $1$ to about $3$. The recurrence's own values sit slightly above
the continuous-time ones ($2.280$ against $2.264$ at $t = 0.25$, $3.030$
against $3.000$ at $t = 3$), the same discretization error as before.

Diffusion models usually let the rate vary with time. The **VP-SDE** of
score-based diffusion [@song2021score],

$$
dX = -\tfrac12 \beta(t)\, X \, dt + \sqrt{\beta(t)}\; dW,
$$

is an OU process with time-dependent rate $\theta(t) = \beta(t)/2$ and noise
amplitude $g(t)^2 = \beta(t)$. It keeps $g^2 = 2\theta$ at every instant, so
the variance stays at $1$ along the whole schedule, and its marginal is
the equation with $\bar{\alpha}_t = e^{-\int_0^t \beta(s)\,ds}$
(problem 10 proves that its transitions are
exact Gaussians). This is the forward process that score matching learns to
reverse, and the variance-preserving path of
that section.

## Simulation

### The Euler--Maruyama Scheme

Most SDEs have no closed-form solution, so we need to simulate them. Our
recurrence already shows how. To simulate the equation on a
grid $t_n = n\,\Delta t$ we
freeze the coefficients over each step, as forward Euler does for ODEs
(that section), and add a Gaussian kick whose size the
increment form the equation dictates. The
**Euler--Maruyama** update [@Maruyama.1955] is

$$
\hat{X}_{n+1} = \hat{X}_n + f(\hat{X}_n, t_n)\,\Delta t
+ g(\hat{X}_n, t_n)\,\sqrt{\Delta t}\;\xi_n,
\qquad \xi_n \sim \mathcal{N}(0, 1) \textrm{ i.i.d.},
$$

where $\hat{X}_n$ approximates $X_{t_n}$ and $\hat X_0 = X_0$. The scheme is a
drift step plus a Brownian increment over the step. The kick scales as
$\sqrt{\Delta t}$, since any other exponent makes the noise vanish or explode
in the limit (that section), and with $g = 0$ the scheme is
forward Euler.

For the OU process the step reads

$$
\hat{X}_{n+1} = (1 - \theta\,\Delta t)\,\hat{X}_n + g\sqrt{\Delta t}\;\xi_n
= 0.99\,\hat{X}_n + 0.1414\,\xi_n ,
$$

where the second form is the running example at $\Delta t = 0.01$, the
recurrence we have been running all along.

For this process we could have done better. With $g^2 = 2\theta$ the kernel
the equation over one interval is
$\mathcal{N}(e^{-\theta\Delta t}x, 1 - e^{-2\theta\Delta t})$, and sampling
from it is the forward step of DDPM
(that section),
$x_k = \sqrt{1 - \beta}\, x_{k-1} + \sqrt{\beta}\,\epsilon$ with
$1 - \beta = e^{-2\theta\Delta t}$. DDPM's step is therefore exact at every
step size, and Euler--Maruyama is its first-order approximation: at
$\theta = 1$ and $\Delta t = 0.01$ the exact step multiplies by
$e^{-0.01} = 0.99005$ and adds noise of variance $1 - e^{-0.02} = 0.0198$,
against $0.99$ and $0.02$. Such exact steps need a linear equation
(problem 10); for a general drift
there is only the approximate scheme.

The helper below implements the equation for a batch of paths at
once. It takes the increments $\Delta W$ as an argument, so that two
simulations can be driven by the same noise.

In [ ]:
def euler_maruyama(f, g, x0, t, dW):
    """Simulate dX = f dt + g dW on the grid t for a batch of paths."""
    X = np.empty((len(x0), len(t)))
    X[:, 0] = x0
    for k in range(len(t) - 1):
        h = t[k + 1] - t[k]
        X[:, k + 1] = (X[:, k] + f(X[:, k], t[k]) * h
                       + g(X[:, k], t[k]) * dW[:, k])
    return X

A quick test of the helper: with the noise switched off it must reproduce
forward Euler on $\dot{x} = -x$.

In [ ]:
t_ode = np.linspace(0, 2.0, 201)
X_ode = euler_maruyama(lambda x, s: -x, lambda x, s: 0.0,
                       np.ones(1), t_ode, np.zeros((1, 200)))
print('g = 0 recovers Euler on dx/dt = -x: '
      f'max |X_t - e^(-t)| = {np.abs(X_ode[0] - np.exp(-t_ode)).max():.5f}')

The error is the Euler error at $h = 10^{-2}$, which is largest near $t = 1$,
where it equals $e^{-1} - 0.99^{100}$. With the noise switched on, "error"
itself needs a definition, and there are two.

### Strong and Weak Convergence

A simulated path is random, so we must say what it means for it to be close
to the true solution. There are two answers, depending on whether we care
about paths or about laws.

Let $N = T/\Delta t$ be the number of steps, so that $\hat{X}_N$ approximates
$X_T$. Drive the scheme and the exact solution with the same Brownian
increments. The **strong error** $\mathbb{E}\,|\hat{X}_N - X_T|$ measures
pathwise tracking: does the simulated trajectory follow the true trajectory
driven by that noise? The **weak error**
$|\mathbb{E}\,\varphi(\hat{X}_N) - \mathbb{E}\,\varphi(X_T)|$, for smooth test
functions $\varphi$, measures only whether the law at time $T$ comes out
right. A generative model needs weak accuracy, since it only uses the samples
at the final time. Strong accuracy matters when paths are coupled across
resolutions, as in multilevel Monte Carlo, which uses cheap coarse paths to
correct expensive fine ones (problem 11).
the figure shows the two notions side by side.

![Strong versus weak convergence of Euler--Maruyama on the running example $dX=-X\,dt+\sqrt2\,dW$ from $X_0=2$ on $[0,1]$. Left: one coarse path ($16$ steps, orange) against a fine reference ($4096$ steps, blue) driven by the same Brownian increments; the strip below magnifies their gap at the coarse grid times, whose size is what strong error measures. Right: terminal histograms of $20{,}000$ paths at the two step sizes nearly match each other and the analytic marginal (black), illustrating weak accuracy; at $16$ steps the terminal variance is about $4\%$ too large.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-dyn-strong-weak.svg)

The convergence rates are classical theorems, and their proofs are beyond
this section; they are in @Kloeden.Platen.1992, together with the
precise smoothness and growth conditions on $f$, $g$, and $\varphi$, which
we only indicate. A scheme has strong order $p$ if its strong error is at
most $C\,\Delta t^{p}$, and weak order $p$ likewise.

**Theorem (orders of Euler--Maruyama; stated without proof).** *Let
$\mathbb{E}[X_0^2] < \infty$ and let $f$ and $g$ be Lipschitz in $x$ with
linear growth and sufficiently regular in $t$.*

1. *Euler--Maruyama has strong order $\tfrac12$:
   $\mathbb{E}\,|\hat{X}_N - X_T| \le C\,\Delta t^{1/2}$.*
2. *If the noise is additive, $g = g(t)$, and the coefficients are smooth
   with bounded derivatives, the strong order is $1$:
   $\mathbb{E}\,|\hat{X}_N - X_T| \le C\,\Delta t$.*
3. *If $f$, $g$, and $\varphi$ are smooth with derivatives of polynomial
   growth and $X_0$ has finite moments of all orders, the weak order is $1$,
   for additive and multiplicative noise alike:
   $|\mathbb{E}\,\varphi(\hat{X}_N) - \mathbb{E}\,\varphi(X_T)| \le C\,\Delta t$.*

A count of single-step errors explains the exponents. It is not a proof: it
adds the errors of the steps as if each stayed where it was made, and the
proofs must also control how the later dynamics carry an early error forward.
The Euler--Maruyama step is the stochastic Taylor expansion of the solution
over one step, truncated after the $g\,\Delta W$ term. For multiplicative
noise the first omitted term is Milstein's
$\tfrac12 g\,\partial_x g\left((\Delta W)^2 - \Delta t\right)$
[@Milstein.1975]. It has mean zero, and its typical size is
$\tfrac12 |g\,\partial_x g|\,\Delta t\; \mathbb{E}|\xi^2 - 1| \approx
0.97 \cdot \tfrac12 |g\,\partial_x g|\,\Delta t$. The $N = T/\Delta t$ steps
contribute independent mean-zero terms of this size, and such terms add like a
random walk, to $\sqrt{N}\,\Delta t = \sqrt{T\,\Delta t}$: order $\tfrac12$.

For additive noise $\partial_x g = 0$ removes this term. The next omitted
stochastic terms have per-step standard deviation of order $\Delta t^{3/2}$ and
add to $\sqrt{N}\,\Delta t^{3/2} = O(\Delta t)$, while the deterministic
remainders of order $\Delta t^2$ add to $N\,\Delta t^2 = O(\Delta t)$: order
$1$. For the weak error only means matter. Every omitted stochastic term has
mean zero, the bias of one step is of order $\Delta t^2$, and $N$ steps give
$O(\Delta t)$, whatever the noise.

We measure both strong orders on the same noise: the running example from
$X_0 = 2$ against a reference on a $4096$-step grid, and geometric Brownian
motion ($\mu = \nu = 1$, $X_0 = 1.5$) against its exact solution
the equation. Coarse increments are sums of consecutive fine ones,
so every resolution sees the same Brownian path.

In [ ]:
T, n_ref, n_em = 1.0, 2**12, 2000
mu_g, nu_g = 1.0, 1.0                        # GBM: multiplicative noise
ou_f, ou_g = lambda x, s: -theta_ou * x, lambda x, s: g_ou
gbm_f, gbm_g = lambda x, s: mu_g * x, lambda x, s: nu_g * x
x0_ou, x0_gbm = np.full(n_em, 2.0), np.full(n_em, 1.5)
dW_f = np.sqrt(T / n_ref) * rng.standard_normal((n_em, n_ref))
t_f = np.linspace(0, T, n_ref + 1)
ou_ref = euler_maruyama(ou_f, ou_g, x0_ou, t_f, dW_f)[:, -1]
gbm_exact = 1.5 * np.exp((mu_g - nu_g**2 / 2) * T + nu_g * dW_f.sum(axis=1))
dts, errs_ou, errs_gbm = [], [], []
for n in [2**k for k in range(3, 9)]:        # 8 to 256 steps
    dW = dW_f.reshape(n_em, n, n_ref // n).sum(axis=2)
    t = np.linspace(0, T, n + 1)
    ou = euler_maruyama(ou_f, ou_g, x0_ou, t, dW)[:, -1]
    gbm = euler_maruyama(gbm_f, gbm_g, x0_gbm, t, dW)[:, -1]
    dts.append(T / n)
    errs_ou.append(np.abs(ou - ou_ref).mean())
    errs_gbm.append(np.abs(gbm - gbm_exact).mean())
print('strong-order slopes:  OU (additive) '
      f'{np.polyfit(np.log(dts), np.log(errs_ou), 1)[0]:.2f},'
      '  GBM (multiplicative) '
      f'{np.polyfit(np.log(dts), np.log(errs_gbm), 1)[0]:.2f}')
d2l.plot(dts, [errs_ou, errs_gbm, np.array(dts), 0.5 * np.sqrt(dts)],
         'step size', 'strong error', xscale='log', yscale='log',
         legend=['OU (additive)', 'GBM (multiplicative)',
                 'slope 1', 'slope 1/2'], figsize=(5, 3))

The measured slopes are close to $1$ for the additive-noise OU process and
close to $\tfrac12$ for the multiplicative-noise geometric Brownian motion,
the two strong orders of the theorem. The two curves have different heights
because the processes have different scales. What can be compared is how
they fall: each halving of the step halves the OU error and divides the
error of geometric Brownian motion by about $\sqrt{2}$.

For the weak order no sampling is needed. Euler--Maruyama applied to the OU
process is a linear Gaussian recursion: if $\hat{X}_n$ is Gaussian, so is
$\hat{X}_{n+1}$. So the simulated marginal is Gaussian at every step, with mean
and variance

$$
m_{n+1} = (1 - \theta\,\Delta t)\,m_n,
\qquad
v_{n+1} = (1 - \theta\,\Delta t)^2\, v_n + g^2\,\Delta t .
$$

The mean is multiplied by $1 - \theta\Delta t$ per step where the exact
solution multiplies by $e^{-\theta\Delta t}$ ($0.900$ against $0.905$ at
$\Delta t = 0.1$). The variance recursion sums a geometric series,
$v_n = g^2\Delta t\,(1 - a^{2n})/(1 - a^2)$ with $a = 1 - \theta\Delta t$
and $v_0 = 0$, whose limit is the fixed point $v_\infty$ found earlier.
Evaluating these closed forms at $T = 1$ from $X_0 = 2$ for six step counts
gives the errors of the simulated law with no sampling error to blur their
slopes.

In [ ]:
mean_T, var_T = 2 * np.exp(-1), 1 - np.exp(-2)  # exact OU kernel at T = 1
print(f'{"n":>4} {"EM mean":>8} {"EM var":>7}'
      f'   exact {mean_T:.4f}, {var_T:.4f}')
dts, errs = [], []
for n in [10, 20, 40, 80, 160, 320]:
    h = 1 / n
    a_h = 1 - theta_ou * h                   # one-step mean multiplier
    m = 2 * a_h**n                           # EM mean after n steps
    v = g_ou**2 * h * (1 - a_h**(2 * n)) / (1 - a_h**2)   # geometric series
    print(f'{n:4d} {m:8.4f} {v:7.4f}')
    dts.append(h)
    errs.append([abs(m - mean_T), abs(v - var_T)])
s_m, s_v = np.polyfit(np.log(dts), np.log(np.array(errs)), 1)[0]
print(f'weak-order slopes:  mean {s_m:.2f},  variance {s_v:.2f}')

At $10$ steps the simulated law is $\mathcal{N}(0.697, 0.925)$ against the
exact $\mathcal{N}(0.736, 0.865)$; at $320$ steps it is
$\mathcal{N}(0.735, 0.866)$. Both fitted slopes are $1.01$: halving the step
halves the error of the marginal, which is weak order $1$.

Multiplicative noise halves the strong order but leaves the weak order at $1$
(problem 6). A scheme that looks more
accurate can also converge to the wrong equation: Heun's method, which
averages the noise coefficient over the step, converges to the Stratonovich
solution instead of the Itô one
(problem 9).

Everything in closed form so far was conditional on the start $x_0$. For
non-Gaussian data the marginal $p_t$ of the whole cloud has no closed form,
and generation needs it: to run the process backward, one must know where
the density is. The next section derives the equation that $p_t$
obeys (that section).

## Summary

Brownian motion is the limit of the $\pm\sqrt{\Delta t}$ coin-flip walk. It has
independent $\mathcal{N}(0, t - s)$ increments, covariance $\min(s, t)$, and
quadratic variation $\sum_i (\Delta W_i)^2 \to t$, and its paths are nowhere
differentiable. Because $(dW)^2 = dt$, the stochastic integral must fix its
evaluation point. Itô's left endpoint gives the zero mean and the isometry
$\mathbb{E}[(\int H\,dW)^2] = \int \mathbb{E}[H^2]\,ds$, and the chain rule
gains the correction $\tfrac12 g^2 \phi_{xx}\,dt$. The Ornstein--Uhlenbeck
process $dX = -\theta X\,dt + g\,dW$ has the Gaussian kernel
$\mathcal{N}\big(x_0 e^{-\theta t}, \tfrac{g^2}{2\theta}(1 - e^{-2\theta t})\big)$
and the stationary law $\mathcal{N}(0, g^2/2\theta)$, reached from every
start; with $g^2 = 2\theta$, unit-variance data keep variance $1$ at every
time. This is the forward process of diffusion models, and the forward step
of DDPM is its exact transition. Euler--Maruyama is Euler plus a
$\sqrt{\Delta t}$ kick, with strong order $1$ for additive noise, $\tfrac12$
for multiplicative noise, and weak order $1$ for both.

## Exercises

1. **Biased random walk.** A walker takes a step of size $\pm c\sqrt{\Delta t}$
   every $\Delta t$ time units, as in that section, but
   its coin is biased: the step is positive with probability
   $p = \tfrac12\left(1 + b\,\Delta t^{\delta}\right)$ for constants
   $b, c, \delta > 0$ and $\Delta t$ small enough that $p \le 1$. Let
   $X^{(\Delta t)}_t$ be the position after $k = t/\Delta t$ steps, starting
   from $0$.
    1. Compute $\mathbb{E}\big[X^{(\Delta t)}_t\big]$ and
       $\textrm{Var}\big(X^{(\Delta t)}_t\big)$. For which exponent
       $\delta$ does the mean have a finite nonzero limit as
       $\Delta t \to 0$? What happens to the mean for smaller and for larger
       $\delta$?
    1. For that exponent, find the limits of the mean and the variance.
       Which constant drift and diffusion coefficients give an SDE
       the equation whose solution from $X_0 = 0$ has the
       same mean and variance at every $t$?
    1. Take $c = 1$ and $\Delta t = 10^{-3}$, and choose $b$ so that the
       drift in (b) equals $1$. Compute $p$. Compare the drift displacement
       with the standard deviation of the noise displacement, first over one
       step and then over the interval $[0, 1]$.

1. **Infinite total variation.** The total variation of a path $x$ along a
   partition $0 = t_0 < t_1 < \cdots < t_n = t$ is
   $V_n = \sum_i |x(t_{i+1}) - x(t_i)|$. Pathwise integrals
   $\int H\,dx$, defined as limits of sums $\sum_i H(s_i)\,\Delta x_i$ with
   arbitrary evaluation points $s_i \in [t_i, t_{i+1}]$, exist for every
   continuous $H$ when $V_n$ stays bounded as the mesh shrinks.
    1. Let $x = W$ and use the uniform partition with $\Delta t_i = t/n$.
       Compute $\mathbb{E}[V_n]$ and $\textrm{Var}(V_n)$, using
       $\mathbb{E}|\xi| = \sqrt{2/\pi}$ for $\xi \sim \mathcal{N}(0, 1)$, and
       conclude that $V_n \to \infty$ in probability as $n \to \infty$.
    1. Let $x$ be continuous with $V_n \le V$ for every partition. Show that
       $\max_i |\Delta x_i| \to 0$ as the mesh shrinks, and deduce
       $\sum_i (\Delta x_i)^2 \le \max_i |\Delta x_i| \cdot V \to 0$: such a
       path has quadratic variation zero. Why does the equation
       then give a second proof that Brownian paths have unbounded total
       variation?
    1. Let $x$ be continuous with bounded total variation. Show that the
       left-endpoint sums $\sum_i x(t_i)\,\Delta x_i$ and the right-endpoint
       sums $\sum_i x(t_{i+1})\,\Delta x_i$ both converge to
       $\tfrac12\left(x(t)^2 - x(0)^2\right)$. Apply the same algebra to
       $x = W$ to derive the equation without Itô's lemma.

1. [code] **Quadratic variation of an Itô process.** Let $X$ solve
   the equation, and let $[X]_t$ be the limit in probability
   of $\sum_i (X_{t_{i+1}} - X_{t_i})^2$ over partitions of $[0, t]$ whose
   mesh tends to zero. The computation of $(dX)^2$ in
   that section suggests
   $[X]_t = \int_0^t g(X_s, s)^2\, ds$.
    1. Write $X_t = X_0 + A_t + M_t$ with $A_t = \int_0^t f(X_s, s)\,ds$ and
       $M_t = \int_0^t g(X_s, s)\,dW_s$, and assume that $f(X_s, s)$ is
       bounded on $[0, t]$. Show that $\sum_i (\Delta A_i)^2$ and
       $\sum_i \Delta A_i\, \Delta M_i$ tend to zero along every path, so
       that $[X]_t = [M]_t$. Problem
       2b supplies the first
       limit; the second needs only the continuity of $M$.
    1. Reshape the increments `dW_qv` of `#sdes-quadratic-variation` to shape
       $(1, 2^{18})$ and pass them to `euler_maruyama` to simulate the OU
       process $dX = -X\,dt + \sqrt{2}\,dW$ from $X_0 = 2$ and the geometric
       Brownian motion $dX = X\,dt + X\,dW$ from $X_0 = 1.5$ on $[0, 1]$.
       For each path, print the sum of squared increments on the subsampled
       grids of that cell next to $\int_0^1 g(X_s, s)^2\,ds$ evaluated along
       the same path. For which of the two processes is $[X]_1$ random?
    1. Let $X_t = \mu t + g W_t$ with constants $\mu$ and $g$ be observed at
       $n + 1$ equally spaced times on $[0, T]$. Show that
       $\frac{1}{T}\sum_i (\Delta X_i)^2 \to g^2$ in mean square as
       $n \to \infty$ for every fixed $T$, whereas the estimator $X_T / T$
       of $\mu$ has variance $g^2 / T$ for every $n$. What does this
       imply for estimating the diffusion coefficient and the drift from
       one finely sampled trajectory?

1. **Geometric Brownian motion and the $-\nu^2/2$ correction.** Consider
   $dX = \mu X\,dt + \nu X\,dW$ with $X_0 > 0$ and its solution, given
   by the equation.
    1. Using $\mathbb{E}\big[e^{\lambda Z}\big] = e^{\lambda^2 v/2}$ for
       $Z \sim \mathcal{N}(0, v)$, show that
       $\mathbb{E}[X_t] = X_0 e^{\mu t}$ and
       $\mathbb{E}[X_t^2] = X_0^2 e^{(2\mu + \nu^2) t}$. Find the median
       of $X_t$.
    1. Take $\mu = 0.1$ and $\nu = 0.6$. What happens to the mean, to the
       median, and to a typical path as $t \to \infty$? How can these
       statements hold at the same time?
    1. For which $\nu$, at a fixed $\mu > 0$, does the median stop growing?
       Check the condition for the parameters of (b) on $100$ paths
       simulated with `euler_maruyama` to $t = 20$ at $\Delta t = 10^{-2}$:
       what fraction of the paths ends below $X_0$, and what fraction does
       the exact solution the equation predict?

   *Adapted from Simo Särkkä and Arno Solin,
   [Applied Stochastic Differential Equations](https://users.aalto.fi/~asolin/sde-book/sde-book.pdf),
   Exercise 4.3.*

1. [code] **Strong order and Milstein's correction.** As
   that section explains, Euler--Maruyama omits the
   Milstein term
   $\tfrac12 g\,\partial_x g\left((\Delta W)^2 - \Delta t\right)$, and its
   strong order is $\tfrac12$ unless that term vanishes.
    1. By the equation, $Y = \log X$ satisfies an SDE with
       additive noise. Apply `euler_maruyama` to that SDE with the
       increments `dW_f` and the step sizes of `#sdes-em-strong-order`,
       exponentiate the endpoints, and measure the strong error against
       `gbm_exact`. Explain the result.
    1. Use Itô's lemma to show that for $dX = f(X)\,dt + g(X)\,dW$ with
       $g > 0$, the variable $Y = \psi(X)$ with $\psi' = 1/g$ has unit
       additive noise, and find its drift.
    1. Milstein's scheme [@Milstein.1975] adds the first omitted term of
       the stochastic Taylor expansion to the Euler--Maruyama step,

       $$\hat X_{n+1} = \hat X_n + f\,\Delta t + g\,\Delta W_n + \tfrac12\, g\,\partial_x g\left((\Delta W_n)^2 - \Delta t\right),$$

       with $f$, $g$, and $\partial_x g$ evaluated at $(\hat X_n, t_n)$.
       Implement it for the geometric Brownian motion of
       `#sdes-em-strong-order`, and measure its strong-order slope on the
       same increments and step sizes. By what factor does it reduce the
       strong error at $\Delta t = 2^{-8}$?

1. [code] **Weak order under multiplicative noise.** For geometric Brownian
   motion, the Euler--Maruyama step multiplies $\hat X_n$ by
   $1 + \mu\,\Delta t + \nu\,\Delta W_n$. Its strong order is
   $\tfrac12$, as `#sdes-em-strong-order` measures.
    1. Derive recursions for the mean and the second moment of the simulated
       $\hat X_n$, and solve them in closed form.
    1. As `#sdes-em-weak-order` does for the OU process, compute the errors
       of both moments at $T = 1$ against the exact moments of problem
       4a, for
       $\mu = \nu = 1$, $X_0 = 1.5$, and
       $n \in \{20, 40, 80, 160, 320, 640\}$ steps, and fit the slopes.
       How do they compare with the strong-order slope of the same scheme?

1. [code] **Covariance of the OU process.** Let $X$ be the OU
   process the equation.
    1. For $s \le t$, split the integral in the equation
       at time $s$ and show that
       $\textrm{Cov}(X_s, X_t \mid X_0) = e^{-\theta(t - s)}\textrm{Var}(X_s \mid X_0)$.
       Write the result in closed form with the equation.
    1. Let $X_0 \sim \mathcal{N}(0, g^2/2\theta)$ be independent of $W$.
       Show that
       $\textrm{Cov}(X_s, X_t) = \frac{g^2}{2\theta}\, e^{-\theta |t - s|}$
       for all $s, t \ge 0$.
    1. Estimate $\textrm{Cov}(X_1, X_t)$ for $t \in \{1, 1.5, 2, 4\}$ from the
       saved clouds `X_ou_snap` of `#sdes-ou-cloud`, and compare the
       estimates with the formulas of (a) and (b). Which formula does the
       cloud follow, and why?
    1. A sampler stores the state of a stationary OU process every $\Delta$
       time units. How large must $\Delta$ be for the correlation between
       consecutive stored states to fall below $0.01$?

1. **Variance preservation, exactly.** Let $X_t$ follow
   the equation with zero-mean data $X_0$.
    1. Assume unit-variance data and show that
       $\mathbb{E}[X_t^4] = 3 + \bar{\alpha}_t^2\left(\mathbb{E}[X_0^4] - 3\right)$.
       Specialize to Rademacher data $X_0 = \pm 1$ to obtain the prediction
       $3 - 2\bar{\alpha}_t^2$ that `#sdes-vp-normalization` prints.
    1. The excess kurtosis $\mathbb{E}[X^4]/\textrm{Var}(X)^2 - 3$ is
       zero for every Gaussian. At what rate in $t$ does it decay for
       $X_t$? Compare this rate with the rate at which the conditional mean
       $\mathbb{E}[X_t \mid X_0]$ forgets the data.
    1. Suppose instead that the data have variance $s^2$. Find
       $\textrm{Var}(X_t)$. For $s = 2$ and $\theta = 1$, how long must
       the process run before $\textrm{Var}(X_T)$ is within $1\%$ of
       the variance of the reference $\mathcal{N}(0, 1)$?

1. [code] **Itô versus Stratonovich.** ● By problem
   2c, the average of the left-
   and right-endpoint sums for $\int_0^t W\,dW$ equals $\tfrac12 W_t^2$ for
   every partition. Sums of this kind define the Stratonovich integral
   [@Stratonovich.1966], which obeys the ordinary chain rule.
    1. Heun's method of that section carries over to
       SDEs if the noise term is treated like a second slope. The predictor
       is the Euler--Maruyama step
       $\tilde{X} = \hat X_n + f(\hat X_n)\,\Delta t + g(\hat X_n)\,\Delta W_n$,
       and the corrector averages both coefficients:

       $$\hat X_{n+1} = \hat X_n + \tfrac12\left[f(\hat X_n) + f(\tilde{X})\right]\Delta t + \tfrac12\left[g(\hat X_n) + g(\tilde{X})\right]\Delta W_n.$$

       Implement it for the geometric Brownian motion of
       `#sdes-em-strong-order`, on that cell's increments `dW_f` and step
       sizes. Compare its endpoints with the exact Itô solution
       the equation and with $X_0 \exp(\mu T + \nu W_T)$. To which
       of the two does the scheme converge?
    1. Expand the corrector's noise term to first order in
       $\tilde{X} - \hat X_n$, and use $(\Delta W_n)^2 \approx \Delta t$ to
       show that the scheme behaves like Euler--Maruyama for an Itô SDE with
       a modified drift. Find that drift and check that it explains the limit
       in (a).
    1. Why do the two conventions give the same solution for every SDE of
       the form the equation?

1. **Exact transitions of linear SDEs.** ● Consider the linear SDE
   $dX = -\theta(t) X\,dt + g(t)\,dW$ with continuous $\theta$ and
   $g$, and let $\Theta(t) = \int_0^t \theta(u)\,du$.
    1. Apply Itô's lemma to $\phi(x, t) = e^{\Theta(t)} x$ and show that,
       given $X_s$, the state $X_t$ at a later time $t$ is Gaussian with mean
       $e^{-(\Theta(t) - \Theta(s))} X_s$ and variance
       $\int_s^t e^{-2(\Theta(t) - \Theta(u))}\, g(u)^2\, du$.
    1. Specialize (a) to the VP-SDE the equation and show that
       a transition of any length maps a unit-variance state to a
       unit-variance state. Conversely, assume for the linear SDE of (a) that
       $\textrm{Var}(X_t) = 1$ for every $t \ge 0$ whenever
       $\textrm{Var}(X_0) = 1$, and show that
       $g(t)^2 = 2\,\theta(t)$ at every $t$.
    1. Consider the semilinear SDE
       $dX = \left(-\theta X + q(X)\right)dt + g\,dW$ with constants
       $\theta, g > 0$ and a smooth function $q$. The exponential
       Euler--Maruyama step holds $q$ at the value $q(\hat X_n)$ over a step
       of length $\Delta t$ and solves the remaining linear SDE exactly, by
       the method of (a). Write the step. Show that it agrees with the
       Euler--Maruyama step the equation to first order in
       $\Delta t$ and that it is exact when $q$ is constant. For
       $q \equiv 0$, Euler--Maruyama damps the mean only for
       $\Delta t < 2/\theta$ (that section). What
       limits the step size of the exponential step?
    1. Exact steps have zero weak error at every step size. Which property
       of the drift and the diffusion of the VP-SDE makes them available?
       The generative sampler of a diffusion model simulates,
       backward in time, an SDE whose drift is
       $-\tfrac12\beta(t)\,x - \beta(t)\,\nabla \log p_t(x)$: the VP drift
       plus a multiple of the score of the forward marginals, which is a
       nonlinear function of $x$ for non-Gaussian data. Why can its steps not
       be made exact? Exponential integrators for diffusion samplers
       [@Zhang.Chen.2023] apply the step of (c) to this drift. Which
       part of the drift does that step treat exactly, and which part does
       it approximate?

1. [extended] **Multilevel Monte Carlo.** ●● Estimating
   $\mathbb{E}[\varphi(X_T)]$ by averaging over Euler--Maruyama paths has two
   sources of error, the weak bias of the scheme and the sampling noise.
   Multilevel Monte Carlo [@Giles.2008] reduces the cost by coupling
   paths across step sizes, which is one of the uses of strong accuracy named
   in that section. Throughout, the mean-square strong
   error $\mathbb{E}\big[(\hat X_N - X_T)^2\big]^{1/2}$ has the same
   orders as the strong error of that section.
    1. Plain Monte Carlo averages $\varphi(\hat X_N)$ over $M$
       independent paths at a cost of $MT/\Delta t$ steps. Assume a weak
       bias $c_1\,\Delta t$ and a variance $v$ of $\varphi(\hat X_N)$.
       Show that a mean squared error below $\varepsilon^2$ costs order
       $\varepsilon^{-3}$ steps at the best choice of $M$ and $\Delta t$.
    1. Let $P_l$ be $\varphi$ of the Euler--Maruyama endpoint with step
       $h_l = T\,2^{-l}$, and write
       $\mathbb{E}[P_L] = \mathbb{E}[P_0] + \sum_{l=1}^{L}\mathbb{E}[P_l - P_{l-1}]$.
       Each term is estimated from $M_l$ independent pairs in which the coarse
       path uses the sums of consecutive pairs of fine increments, as in
       `#sdes-em-strong-order`. For Lipschitz $\varphi$, bound
       $V_l = \textrm{Var}(P_l - P_{l-1})$ by the mean-square strong
       error. Take $\varphi(x) = x$, and predict how $V_l$ scales with $h_l$
       for the OU process and the geometric Brownian motion of
       `#sdes-em-strong-order`, with that cell's parameters. Measure $V_l$
       for $l = 1, \ldots, 8$ with `euler_maruyama`, and fit the slopes on
       the finer levels.
    1. Minimize the total cost $\sum_l M_l / h_l$ subject to
       $\sum_l V_l / M_l \le \varepsilon^2 / 2$ with a Lagrange multiplier.
       Choose the finest level $L$ so that the bias stays below
       $\varepsilon / \sqrt{2}$. Assuming $V_l \propto h_l^{r}$, show that
       the cost is of order $\varepsilon^{-2}$ for $r = 2$ and of order
       $\varepsilon^{-2} (\log \varepsilon)^2$ for $r = 1$.
    1. Implement both estimators for $\mathbb{E}[X_T] = X_0 e^{\mu T}$ of the
       geometric Brownian motion of `#sdes-em-strong-order`, with $M_l$ from
       (c) and your measured $V_l$. For
       $\varepsilon \in \{0.2, 0.1, 0.05, 0.02\}$, record the total number of
       steps each estimator uses and check its error over a few repetitions.
       Plot $\varepsilon^2$ times the cost against $\varepsilon$. Repeat with
       the Milstein scheme of problem 5c
       in place of Euler--Maruyama. `euler_maruyama` stores every path in
       full, so simulate large $M_l$ in batches.
    1. Repeat the measurement of (b) for the discontinuous
       $\varphi(x) = \mathbf{1}(x > X_0)$. Why does $V_l$ decay more slowly,
       and what does the slower decay do to the cost?

1. [code] **Brownian scaling and exit times.**
    1. For $c > 0$ let $\tilde{W}_t = W_{ct}/\sqrt{c}$. Check properties 1–4
       of that section and conclude that $\tilde{W}$ is a
       Brownian motion. Which property of the $\pm\sqrt{\Delta t}$ walk does
       this make exact?
    1. Let $\tau_a$ be the first time $|W_t|$ reaches a level $a > 0$. Use (a)
       to show that $\tau_a$ has the same distribution as $a^2 \tau_1$, so
       that $\mathbb{E}[\tau_a] = a^2\, \mathbb{E}[\tau_1]$.
    1. Granting $\mathbb{E}[\tau_1] = 1$ (it follows from
       $\mathbb{E}[W_{\tau}^2] = \mathbb{E}[\tau]$, a consequence of
       the equation and the zero mean of the Itô integral up to a
       stopping time), simulate $4{,}000$ walks with Gaussian steps of
       variance $\Delta t = 10^{-3}$ and estimate $\mathbb{E}[\tau_a]$ for
       $a = 0.5, 1, 2$, with Monte Carlo standard errors. The estimates tend
       to run slightly above $a^2$, most clearly for small $a$. Explain the
       sign of the bias from the way a discrete walk crosses a level, and say
       how it scales with $a$ and $\Delta t$.

[Discussions](https://d2l.discourse.group/t/36838)